# Contract-Year Feature Matrix Integration

**Prepare contract-year base table**

The contract-year table is the analytical backbone of the framework.  
Each row represents one contract in one observation year.

This table is enriched step by step using:
- supplier bridge
- financial indicators
- ESG indicators
- Supplier news with sentiment
- Stocks (later)
- Spend (later)
- Indexes (later)


In [6]:
import pandas as pd
import numpy as np

Load core tables

In [7]:
# Contract base table
contract_year = pd.read_csv(
    "/Users/annita/Desktop/Thesis/tables/joining/contract_year_clean.csv"
)

# Supplier bridge
df_suppliers_bridge = pd.read_csv(
    "/Users/annita/Desktop/Thesis/tables/suppliers/dim_supplier.csv"
)

# Financials
df_financials_clean = pd.read_csv(
    "/Users/annita/Desktop/Thesis/tables/joining/financials_clean.csv"
)

# ESG
df_esg_yearly = pd.read_csv(
    "/Users/annita/Desktop/Thesis/tables/joining/esg_yearly.csv"
)

# News
df_news = pd.read_csv(
    "/Users/annita/Desktop/Thesis/tables/joining/news.csv"
)

**`Contract-Year Base Table`**

The contract-year table represents the core analytical dataset of the framework.

Each row corresponds to a single contract observed in a specific year.

```text
Unit of analysis:
contract × observation_year

In [13]:
df_contract = contract_year.copy()

# Standardize supplier_number
df_contract["supplier_number"] = (
    df_contract["supplier_number"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
)

# Ensure observation_year is numeric
df_contract["observation_year"] = pd.to_numeric(
    df_contract["observation_year"],
    errors="coerce"
).astype("Int64")

print(df_contract.shape)
print(
    "Duplicate contract-year rows:",
    df_contract.duplicated(["contract_id", "observation_year"]).sum()
)

(9201, 35)
Duplicate contract-year rows: 0


In [14]:
df_contract.head()

,contract_id,contract_number,contract_name,contract_status,terminated,term_type,start_date,expiration_date,supplier_id,supplier_number,...,Preferred_supplier_tag,start_year,expiry_year,open_ended_contract,end_year,start_year_capped,observation_year,years_to_expiry,contract_age_years,expiry_pressure_bucket
0,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,0.0,2018,2027,False,2025,2018,2018,9,0,5y_plus
1,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,0.0,2018,2027,False,2025,2018,2019,8,1,5y_plus
2,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,0.0,2018,2027,False,2025,2018,2020,7,2,5y_plus
3,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,0.0,2018,2027,False,2025,2018,2021,6,3,5y_plus
4,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,0.0,2018,2027,False,2025,2018,2022,5,4,3_to_5y


**`Supplier Bridge Table (dim_supplier)`**

The supplier bridge table is used to harmonize internal supplier identifiers with external company identifiers (moodys_bvd_id).

The contract dataset uses an internal identifier:

```text
supplier_number

In [15]:
df_bridge = df_suppliers_bridge.copy()

df_bridge["supplier_key"] = (
    df_bridge["supplier_key"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
)

df_bridge["moodys_bvd_id"] = (
    df_bridge["moodys_bvd_id"]
    .astype("string")
    .str.strip()
)

bridge_cols = ["supplier_key", "moodys_bvd_id"]

df_bridge_slim = df_bridge[bridge_cols].drop_duplicates()

print(
    "Duplicate bridge supplier_key rows:",
    df_bridge_slim.duplicated(["supplier_key"]).sum()
)

Duplicate bridge supplier_key rows: 0


In [16]:
df_suppliers_bridge.head()

,composite_key,supplier_number_name,supplier_key,supplier_name,coupa_name,coupa_supplier_id,coupa_supplier_number,coupa_supplier_name,coupa_created_at,coupa_created_by,...,coupa_parent_company_id,coupa_parent_company_name,coupa_supplier_tag,moodys_bvd_id,moodys_company_name,moodys_status,moodys_closing_date_last_avail_year,moodys_nom_last_avail_year,moodys_financial_risk,moodys_last_closing_year
0,MOODYS_DK71302814-COUPA_12286,12286 - BECTON DICKINSON DENMARK A/S,12286,BECTON DICKINSON DENMARK A/S,12286,7951,12286,BECTON DICKINSON DENMARK A/S,2019-06-12 17:19:54,sftp_integration@coupa.com,...,Moody's supplier,Moody's supplier,SSIMS,DK71302814,Becton Dickinson Denmark A/S,Active,2025-09-30,12,Go ahead,2025
1,MOODYS_GB08336235-COUPA_34339,34339 - P\S\L GROUP EUROPE LIMITED,34339,P\S\L GROUP EUROPE LIMITED,34339,18091,34339,P\S\L GROUP EUROPE LIMITED,2019-06-14 09:47:41,sftp_integration@coupa.com,...,Moody's supplier,Moody's supplier,Moody's supplier,GB08336235,PSL Administration Services UK Limited,Active,2024-12-31,12,Do not source,2024
2,MOODYS_DK30527852-COUPA_40743,40743 - EMENDO CONSULTING GROUP A/S,40743,EMENDO CONSULTING GROUP A/S,40743,11238,40743,EMENDO CONSULTING GROUP A/S,2019-06-14 09:33:00,sftp_integration@coupa.com,...,Moody's supplier,Moody's supplier,IES,DK30527852,Emendo Consulting Group A/S,Active,2024-12-31,12,Do not source,2024
3,MOODYS_DK19993345-COUPA_11855,11855 - BUCH & HOLM A/S,11855,BUCH & HOLM A/S,11855,1712,11855,BUCH & HOLM A/S,2019-06-12 16:38:00,sftp_integration@coupa.com,...,Moody's supplier,Moody's supplier,other,DK19993345,Buch & Holm A/S,Active,2025-09-30,12,Take caution,2025
4,MOODYS_DE6070345525-COUPA_1000976,1000976 - IQVIA COMMERCIAL GMBH & CO. OHG,1000976,IQVIA COMMERCIAL GMBH & CO. OHG,1000976,14457,1000976,IQVIA COMMERCIAL GMBH & CO. OHG,2019-06-14 09:39:31,sftp_integration@coupa.com,...,Moody's supplier,Moody's supplier,Moody's supplier,DE6070345525,IQVIA Commercial GmbH & Co. OHG,Active,2023-12-31,12,Take caution,2023


**`Join Logic: Contract Table and Supplier Bridge`**

The supplier bridge is joined to the contract-year table using a left join.

```text
LEFT JOIN
ON contract_year.supplier_number = dim_supplier.supplier_key

In [17]:
df_contract_bvd = df_contract.merge(
    df_bridge_slim,
    left_on="supplier_number",
    right_on="supplier_key",
    how="left"
).drop(columns=["supplier_key"])

In [19]:
total = len(df_contract_bvd)
matched = df_contract_bvd["moodys_bvd_id"].notna().sum()

print(f"Total contract rows   : {total}")
print(f"Matched to BvD ID     : {matched} ({100*matched/total:.1f}%)")
print(f"No BvD ID             : {total - matched}")

print(
    "Duplicate contract-year rows after bridge:",
    df_contract_bvd.duplicated(["contract_id", "observation_year"]).sum()
)

Total contract rows   : 9201
Matched to BvD ID     : 4634 (50.4%)
No BvD ID             : 4567
Duplicate contract-year rows after bridge: 0


**`Financial Dataset Preparation & Join to contracts`**

The financial dataset contains company-level accounting and risk indicators obtained from Moody’s.

The company identifier:

```text
moodys_bvd_id


Logic

Financial indicators are integrated into the contract-year dataset using the Moody’s company identifier and the contract observation year.

The integration follows this logic:

moodys_bvd_id
+
observation_year
=
Join_Year

In [22]:
df_financials_clean.head()

,moodys_bvd_id,moodys_company_name,Join_Year,closing_year,closing_date,created_at_utc,Status,Implied_rating,risk_level,Financial_level,...,Total_liabilities_Equity_ratio,Long_term_liabilities_Equity_ratio,Short_term_liabilities_Equity_ratio,Interest_coverage_ratio,Solvency_ratio_Asset_based,Debt_Asset_ratio,ROE_using_Net_income,ROA_using_Net_income,Net_assets_turnover,Number_of_employees
0,GB02490104,Bio-Techne LTD,2022,2022.0,2022-06-30,2026-03-05 09:02:26.421112+00:00,Active,Baa,Take caution,NaN,...,31.132483,1.348341,29.784142,42.079646,76.259,0.195834,29.641,22.604,0.919,150.0
1,DE2110000553,Sartorius AG,2022,2022.0,2022-12-31,2026-03-05 09:02:26.421112+00:00,Active,A,Go ahead,Low financial risk,...,162.428072,94.603031,67.825040,16.307692,38.106,0.359717,23.651,9.013,0.811,15942.0
2,LULB25789,SanisSure S.A.,2021,2021.0,2021-12-31,2026-03-05 09:02:26.421112+00:00,Active,Baa,Take caution,NaN,...,272.158195,237.538807,34.619388,NaN,26.870,0.000000,NaN,NaN,NaN,NaN
3,DE7130049291,Amcor Flexibles Singen GmbH,2022,2022.0,2022-06-30,2026-03-05 09:02:26.421112+00:00,Active,Ba,Take caution,NaN,...,383.633361,73.990382,309.642979,9.423212,20.677,0.000000,NaN,NaN,4.936,1242.0
4,LULB186284,Amazon Web Services EMEA SARL,2022,2022.0,2022-12-31,2026-03-05 09:02:26.421112+00:00,Active,Baa,Go ahead,Low financial risk,...,236.155083,0.831076,235.324008,0.812609,29.748,0.000000,22.206,6.606,8.358,NaN


In [24]:
df_fin = df_financials_clean.copy()

df_fin["moodys_bvd_id"] = (
    df_fin["moodys_bvd_id"]
    .astype("string")
    .str.strip()
)

df_fin["Join_Year"] = pd.to_numeric(
    df_fin["Join_Year"],
    errors="coerce"
).astype("Int64")

print(
    "Duplicate financial company-year:",
    df_fin.duplicated(["moodys_bvd_id", "Join_Year"]).sum()
)

# Prefix all non-key columns
fin_key_cols = ["moodys_bvd_id", "Join_Year"]
fin_other_cols = [c for c in df_fin.columns if c not in fin_key_cols]

df_fin_prefixed = df_fin.rename(
    columns={c: f"fin_{c}" for c in fin_other_cols}
)

Duplicate financial company-year: 0


In [25]:
df_contract_fin = df_contract_bvd.merge(
    df_fin_prefixed,
    left_on=["moodys_bvd_id", "observation_year"],
    right_on=["moodys_bvd_id", "Join_Year"],
    how="left"
).drop(columns=["Join_Year"])

In [29]:
print("Rows:", len(df_contract_fin))

print(
    "Duplicate contract-year rows:",
    df_contract_fin.duplicated(
        ["contract_id", "observation_year"]
    ).sum()
)

print(
    "Financial matches:",
    df_contract_fin["fin_Operating_revenue_Turnover_th_DKK"]
    .notna()
    .sum()
)

print("Total columns:", len(df_contract_fin.columns))

Rows: 9201
Duplicate contract-year rows: 0
Financial matches: 1281
Total columns: 87


In [27]:
df_contract_fin.head()

,contract_id,contract_number,contract_name,contract_status,terminated,term_type,start_date,expiration_date,supplier_id,supplier_number,...,fin_Total_liabilities_Equity_ratio,fin_Long_term_liabilities_Equity_ratio,fin_Short_term_liabilities_Equity_ratio,fin_Interest_coverage_ratio,fin_Solvency_ratio_Asset_based,fin_Debt_Asset_ratio,fin_ROE_using_Net_income,fin_ROA_using_Net_income,fin_Net_assets_turnover,fin_Number_of_employees
0,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**`ESG Indicators Integration`**

The ESG dataset is integrated into the contract-year table using the Moody’s company identifier (moodys_bvd_id) and the lagged temporal key (Join_Year).

Join logic

The ESG join follows this structure:

```text
moodys_bvd_id
+
observation_year
=
Join_Year

In [30]:
df_esg = df_esg_yearly.copy()

df_esg["moodys_bvd_id"] = (
    df_esg["moodys_bvd_id"]
    .astype("string")
    .str.strip()
)

df_esg["Join_Year"] = pd.to_numeric(
    df_esg["Join_Year"],
    errors="coerce"
).astype("Int64")

print(
    "Duplicate ESG company-year:",
    df_esg.duplicated(["moodys_bvd_id", "Join_Year"]).sum()
)

# Prefix columns
esg_key_cols = ["moodys_bvd_id", "Join_Year"]
esg_other_cols = [c for c in df_esg.columns if c not in esg_key_cols]

df_esg_prefixed = df_esg.rename(
    columns={c: f"esg_{c}" for c in esg_other_cols}
)

# Join ESG
df_contract_fin_esg = df_contract_fin.merge(
    df_esg_prefixed,
    left_on=["moodys_bvd_id", "observation_year"],
    right_on=["moodys_bvd_id", "Join_Year"],
    how="left"
).drop(columns=["Join_Year"])

Duplicate ESG company-year: 0


In [31]:
print("Rows:", len(df_contract_fin_esg))

print(
    "Duplicate contract-year rows:",
    df_contract_fin_esg.duplicated(
        ["contract_id", "observation_year"]
    ).sum()
)

print(
    "ESG matches:",
    df_contract_fin_esg["esg_esg_overall"].notna().sum()
)

print("Total columns:", len(df_contract_fin_esg.columns))

Rows: 9201
Duplicate contract-year rows: 0
ESG matches: 1012
Total columns: 101


`Overall snapshot`

In [32]:
total = len(df_contract_fin_esg)

bvd_match = df_contract_fin_esg["moodys_bvd_id"].notna().sum()
fin_match = df_contract_fin_esg["fin_Operating_revenue_Turnover_th_DKK"].notna().sum()
esg_match = df_contract_fin_esg["esg_esg_overall"].notna().sum()

print(f"Total contract rows   : {total}")
print(f"Matched to BvD ID     : {bvd_match} ({100*bvd_match/total:.1f}%)")
print(f"Matched to Financials : {fin_match} ({100*fin_match/total:.1f}%)")
print(f"Matched to ESG        : {esg_match} ({100*esg_match/total:.1f}%)")

Total contract rows   : 9201
Matched to BvD ID     : 4634 (50.4%)
Matched to Financials : 1281 (13.9%)
Matched to ESG        : 1012 (11.0%)


**`News Sentiment Integration`**

The news dataset provides supplier-level signals derived from external media coverage (Google-News).

Each observation represents aggregated news sentiment and article counts for a supplier in a specific year.


Join logic

Unlike financial and ESG datasets, the news dataset is linked directly to suppliers using the internal supplier identifier (supplier number).

The join follows this structure:

```text
supplier_number
+
observation_year
=
Join_Year

In [35]:
df_news_view = df_news.copy()

df_news_view["supplier_number"] = (
    df_news_view["supplier_number"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
)

df_news_view["Join_Year"] = pd.to_numeric(
    df_news_view["Join_Year"],
    errors="coerce"
).astype("Int64")

print(
    "Duplicate news supplier-year:",
    df_news_view.duplicated(
        ["supplier_number", "Join_Year"]
    ).sum()
)

# Prefix
news_key_cols = ["supplier_number", "Join_Year"]

news_other_cols = [
    c for c in df_news_view.columns
    if c not in news_key_cols
]

df_news_prefixed = df_news_view.rename(
    columns={c: f"news_{c}" for c in news_other_cols}
)

# Merge the news
df_feature_matrix = df_contract_fin_esg.merge(
    df_news_prefixed,
    left_on=["supplier_number", "observation_year"],
    right_on=["supplier_number", "Join_Year"],
    how="left"
).drop(columns=["Join_Year"])

Duplicate news supplier-year: 0


In [36]:
total = len(df_feature_matrix)

bvd_match = df_feature_matrix["moodys_bvd_id"].notna().sum()
fin_match = df_feature_matrix["fin_Operating_revenue_Turnover_th_DKK"].notna().sum()
esg_match = df_feature_matrix["esg_esg_overall"].notna().sum()
news_match = df_feature_matrix["news_article_count"].notna().sum()

print(f"\n{'='*50}")
print(f"Total contract rows   : {total}")
print(f"Matched to BvD ID     : {bvd_match} ({100*bvd_match/total:.1f}%)")
print(f"Matched to Financials : {fin_match} ({100*fin_match/total:.1f}%)")
print(f"Matched to ESG        : {esg_match} ({100*esg_match/total:.1f}%)")
print(f"Matched to News       : {news_match} ({100*news_match/total:.1f}%)")
print(f"{'='*50}")

print(
    "Duplicate contract-year rows:",
    df_feature_matrix.duplicated(
        ["contract_id", "observation_year"]
    ).sum()
)

print("Total columns:", len(df_feature_matrix.columns))


Total contract rows   : 9201
Matched to BvD ID     : 4634 (50.4%)
Matched to Financials : 1281 (13.9%)
Matched to ESG        : 1012 (11.0%)
Matched to News       : 1185 (12.9%)
Duplicate contract-year rows: 0
Total columns: 110


**` Integrated Contract-Year Feature Matrix`**

External datasets have been successfully integrated into the contract-year base table to create a unified analytical dataset.

The final dataset contains supplier-level risk signals from multiple domains, including financial performance, sustainability indicators, and external media coverage.

**Integrated Data Sources**

The following datasets were merged into the contract-year table:

```text
Supplier bridge (identifier harmonization)
Financial indicators
ESG indicators
News sentiment signals

**Data Integrity**

The left join ensures that:

- all contract-year observations remain in the dataset
- no contracts are dropped due to missing ESG information
- the unit of analysis remains consistent

Missing values will be handled during the feature engineering and modeling stages.



In [37]:
df_feature_matrix.head()

,contract_id,contract_number,contract_name,contract_status,terminated,term_type,start_date,expiration_date,supplier_id,supplier_number,...,esg_industry_max,news_article_count,news_negative_count,news_positive_count,news_neutral_count,news_avg_sentiment_score,news_avg_relevance_score,news_max_relevance_score,news_negative_ratio,news_has_high_relevance_negative_news
0,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
